# 01.04 昇腾云迁移训练

## 本节概述

<table style="text-align: left; margin-left: 0;">
<tr><td align="left"><b>前置要求</b></td><td align="left">已完成 01.03 YOLO 推理</td></tr>
<tr><td align="left"><b>本节目标</b></td><td align="left">在昇腾 NPU 环境完成水果数据集迁移训练，评估模型效果</td></tr>
<tr><td align="left"><b>本节内容</b></td><td align="left">环境准备 → 数据集与训练配置 → 推理与效果评估</td></tr>
</table>

> ⏱️ 训练 50 epoch 在昇腾 NPU 上约 10-30 分钟。

## 第一部分：昇腾 NPU 环境准备

迁移训练对算力要求较高，本节在**昇腾 NPU 环境**（CANNLab 云开发环境）下完成。

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">项目</th><th align="left">要求</th></tr>
<tr><td align="left">CANNLab 模板</td><td align="left"><code>cann_8.5.2-py3.11-A2-arm</code>（选择 CANN 8.5.2 环境）</td></tr>
<tr><td align="left">规格</td><td align="left"><code>1*NPU 910B3 16vCPUs 32GiB</code></td></tr>
<tr><td align="left">Python 内核</td><td align="left"><b>Python 3.11.4 (CANN)</b>（菜单 Kernel → 选择该内核）</td></tr>
<tr><td align="left">关键依赖</td><td align="left">ultralytics + <code>torch_npu</code>（让 PyTorch 识别昇腾 NPU）</td></tr>
</table>

> 💡 训练代码会自动检测设备（`npu:0` → `0`(GPU) → `cpu`）。若无 NPU/GPU 会回退 CPU（仅用于流程验证，速度很慢）。

In [ ]:
# 安装 Ultralytics（CANNLab 环境无需处理依赖冲突）
!pip install ultralytics -i https://pypi.tuna.tsinghua.edu.cn/simple

# 昇腾 NPU 适配：torch_npu 通常已随 CANNLab 镜像预装，无需重复安装。
# 若提示 "No module named 'torch_npu'"，请确认选择了 CANN 8.5.2 的 NPU 模板。
# pip install torch_npu  # 取消注释以手动安装（版本须与 CANN 对齐）

# 若 opencv 提示 numpy 版本警告，可指定兼容版本
# !pip install "numpy<2.0.0" -q

In [ ]:
# 验证环境：确认依赖安装正确
import numpy as np
import matplotlib
import ultralytics
from ultralytics import YOLO

print("NumPy 版本:", np.__version__)
print("Matplotlib 版本:", matplotlib.__version__)
print("Ultralytics 版本:", ultralytics.__version__)
print("✅ 环境就绪，可开始迁移训练")


如果上方版本号显示正常且无报错，说明环境准备完成。

---

## 本节练习

**练习 1（选择）**：在 CANNLab 的 Python 3.11.4 (CANN) 环境下，训练时为何要传 `amp=False` 参数？
- A. 因为 NPU 不支持 AMP，传了会报错
- B. 因为 ultralytics 的 check_amp 会硬编码调 torch.cuda，NPU 版 PyTorch 无 CUDA 模块会崩溃
- C. 需要，必须手动降级 numpy
- D. 不确定

**练习 2（填空）**：迁移学习相比从零训练，优势是 ______ 和 ______。

**练习 3（简答）**：为什么 `patience=10` 参数推荐开启？

In [ ]:
# 查看本节练习答案
!cat ./answer/01.04_transfer_learning/answers_env.txt



---

## 第二部分：水果数据集与训练配置


In [ ]:
# 查看 fruit.yaml 配置文件内容
!cat ./src/fruit.yaml


这个配置说明：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">字段</th><th align="left">含义</th></tr>
<tr><td align="left"><code>path</code></td><td align="left">数据集根目录（相对路径）</td></tr>
<tr><td align="left"><code>train</code></td><td align="left">训练图片相对路径</td></tr>
<tr><td align="left"><code>val</code></td><td align="left">验证图片相对路径</td></tr>
<tr><td align="left"><code>names</code></td><td align="left">类别编号→名称映射（0-4 共 5 类水果）</td></tr>
</table>

> 💡 完整的 fruit.yaml 已放在本小节的 `src/` 目录。实际训练时需要把 `fruit/` 数据集也准备好（含图片和标注）。

## 1. 启动迁移训练

核心代码只有 3 行——加载预训练模型，调用 `train()`：

In [ ]:
# ===== 检查水果数据集是否存在，不存在则从 ModelScope 下载 =====
import os
if not os.path.exists("./src/fruit/images/train"):
    print("⏳ 检测到本地无水果数据集，从 ModelScope 下载（约 155MB）...")
    try:
        import modelscope
    except ImportError:
        import subprocess
        subprocess.check_call(['pip', 'install', 'modelscope', '-q'])
    from modelscope.hub.snapshot_download import snapshot_download
    snapshot_download('Kumako/yolo_fruit_dataset', repo_type='dataset', local_dir='./src/fruit_tmp')
    # 移动文件到正确位置
    import shutil
    os.makedirs('./src/fruit', exist_ok=True)
    for item in os.listdir('./src/fruit_tmp'):
        s = os.path.join('./src/fruit_tmp', item)
        d = os.path.join('./src/fruit', item)
        if os.path.exists(d):
            shutil.rmtree(d) if os.path.isdir(d) else os.remove(d)
        shutil.move(s, d)
    shutil.rmtree('./src/fruit_tmp', ignore_errors=True)
    print("✅ 水果数据集已就绪")
else:
    print("✅ 本地已有水果数据集")

# ===== 昇腾 NPU 兼容补丁 =====
# ultralytics 多处硬编码调用 torch.cuda API（check_amp / _get_memory / _clear_memory），
# 而 NPU 版 PyTorch 不含 CUDA 模块，会在训练验证、显存清理时崩溃。
# 这里用 monkey-patch 把这些函数替换成 NPU 安全版本。
import torch
_is_npu_env = False
try:
    import torch_npu  # noqa: F401
    if torch.npu.is_available():
        _is_npu_env = True
except ImportError:
    pass

if _is_npu_env:
    from ultralytics.engine import trainer as _ult_trainer
    from ultralytics.utils import checks as _ult_checks

    # 1) check_amp：直接返回 True（amp=False 时本就不会真用，但函数仍会被定义）
    _ult_checks.check_amp = lambda model=None: True

    # 2) _get_memory：NPU 上用 torch.npu 查显存，fraction 返回 0（表示显存充足，不清缓存）
    def _npu_get_memory(self, fraction=True):
        try:
            used = torch.npu.memory_allocated()
            total = torch.npu.get_device_properties(0).total_memory
        except Exception:
            used, total = 0, 1
        if fraction:
            return (used / total) if total > 0 else 0
        return used / (1024 ** 3)
    _ult_trainer.BaseTrainer._get_memory = _npu_get_memory

    # 3) _clear_memory：NPU 上只做 gc + empty_cache，跳过 CUDA 相关逻辑
    def _npu_clear_memory(self, threshold=None):
        import gc
        gc.collect()
        try:
            torch.npu.empty_cache()
        except Exception:
            pass
    _ult_trainer.BaseTrainer._clear_memory = _npu_clear_memory

    print("✅ 已加载 NPU 兼容补丁（check_amp / _get_memory / _clear_memory）")
# ===== 补丁结束 =====

from ultralytics import YOLO
import torch

# ===== 自动选择训练设备：昇腾 NPU(CANNLab) → NVIDIA GPU → CPU =====
# 新版 ultralytics 已原生支持 Ascend NPU，只需 import torch_npu 即可识别 "npu" 设备
device = None
is_npu = False
try:
    import torch_npu  # noqa: F401  让 PyTorch / ultralytics 识别昇腾 NPU
    if torch.npu.is_available():
        device = "npu:0"
        is_npu = True
        print(f"✅ 检测到昇腾 NPU: {torch.npu.get_device_name(0)}，将用 NPU 训练")
except ImportError:
    pass
if device is None and torch.cuda.is_available():
    device = 0  # NVIDIA GPU
    print(f"✅ 检测到 NVIDIA GPU: {torch.cuda.get_device_name(0)}，将用 GPU 训练")
if device is None:
    device = "cpu"
    print("⚠️ 未检测到加速卡，回退 CPU 训练（速度会很慢，建议使用 NPU/GPU 环境）")

# 加载预训练模型（迁移训练的基础）
# 注意：指定本地路径，否则会重新下载
model = YOLO("yolov10n.pt")

# 启动迁移训练
# ⚠️ 昇腾 NPU 上的关键参数说明：
#   amp=False —— ultralytics 内部 check_amp() 会硬编码调用 torch.cuda API，
#   而 NPU 版 PyTorch 不含 CUDA，会抛 "Torch not compiled with CUDA enabled"。
#   关闭 AMP（混合精度）即可绕过该 bug，NPU 上 AMP 支持本就不完善，关闭更稳。
#   workers=0 —— CANNLab 云环境 /dev/shm（共享内存）很小，DataLoader 默认多进程
#   会因 "No space left on device" / "Bus error" 崩溃。workers=0 改单进程加载，稳定不崩。
model.train(
    data='./src/fruit.yaml',     # 数据配置文件
    epochs=50,                    # 训练轮次（整个数据集过 50 遍）
    imgsz=640,                    # 训练图像尺寸
    batch=4,                      # 每批 4 张图（云环境显存/共享内存有限，调小更稳）
    device=device,                # ★ 训练设备：npu:0 / 0(GPU) / cpu
    amp=not is_npu,               # ★ NPU 关闭 AMP（绕过 check_amp 的 CUDA 硬编码 bug），GPU 正常开
    workers=0,                    # ★ 单进程数据加载（避免云环境 shm 不足导致 worker 崩溃）
    patience=10,                  # 早停：连续 10 个 epoch 验证指标无提升则停止
    save=True,                    # 保存训练过程中的权重
    name='v10_run1'
)

### 训练过程说明

训练启动后，你会看到类似输出：

```text
Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances    Size
  1/50    ...        ...        ...        ...        ...        640
  2/50    ...        ...        ...        ...        ...        640
  ...
```

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">指标</th><th align="left">含义</th><th align="left">期望趋势</th></tr>
<tr><td align="left"><code>box_loss</code></td><td align="left">边界框回归损失</td><td align="left">逐步下降</td></tr>
<tr><td align="left"><code>cls_loss</code></td><td align="left">分类损失</td><td align="left">逐步下降</td></tr>
<tr><td align="left"><code>dfl_loss</code></td><td align="left">分布焦点损失</td><td align="left">逐步下降</td></tr>
<tr><td align="left"><code>patience</code></td><td align="left">早停计数器</td><td align="left">验证指标无提升时累加</td></tr>
</table>

训练结果保存在 `runs/detect/v10_run1/`下。

## 2. 关键训练超参详解

<table style="text-align: left; margin-left: 0;">
<tr><th align="left">参数</th><th align="left">本实验</th><th align="left">含义与调参建议</th></tr>
<tr><td align="left"><code>epochs</code></td><td align="left">50</td><td align="left">训练轮次。数据量小时 30-50 够用；数据量大时 100-300</td></tr>
<tr><td align="left"><code>imgsz</code></td><td align="left">640</td><td align="left">训练图像尺寸。越大越准但越慢，常用 416/640/1280</td></tr>
<tr><td align="left"><code>batch</code></td><td align="left">8</td><td align="left">批大小。显存不够时调小（如 4 或 2），用 <code>batch=-1</code> 让 YOLO 自动选</td></tr>
<tr><td align="left"><code>patience</code></td><td align="left">10</td><td align="left">早停耐心值。设 0 关闭早停；设 10 表示 10 epoch 无进步就停</td></tr>
<tr><td align="left"><code>data</code></td><td align="left">fruit.yaml</td><td align="left">数据配置文件路径</td></tr>
</table>

## 3. 训练结果产物

训练完成后，`runs/detect/v10_run1/` 目录下会有：

<table style="text-align: left; margin-left: 0;">
<tr><th align="left">文件</th><th align="left">说明</th></tr>
<tr><td align="left"><code>weights/best.pt</code></td><td align="left">验证指标最好的权重（⭐ 推理时用这个）</td></tr>
<tr><td align="left"><code>weights/last.pt</code></td><td align="left">最后一轮的权重</td></tr>
<tr><td align="left"><code>results.png</code></td><td align="left">训练过程 loss 和指标曲线</td></tr>
<tr><td align="left"><code>confusion_matrix.png</code></td><td align="left">混淆矩阵（看哪些类别易混淆）</td></tr>
<tr><td align="left"><code>BoxPR_curve.png</code></td><td align="left">PR 曲线（精度-召回）</td></tr>
</table>

<img src="./images/training_results.png" width="500">

上图是训练过程的 loss 和指标曲线，可以看到 loss 逐步下降、mAP 逐步上升，说明训练正常。

下一节我们加载 `best.pt`，看看训练后的模型检测效果如何。

---

## 本节练习

**练习 1（选择）**：YOLO 标注文件中，边界框坐标 `0.45 0.62 0.30 0.40` 的含义是？
- A. 像素坐标 (45, 62, 30, 40)
- B. 归一化比例：中心点(45%, 62%)，宽30%，高40%
- C. 比例 0.45%、0.62%、0.30%、0.40%
- D. 相对左上角的偏移量

**练习 2（填空）**：训练结果中，推理时应该加载 `weights/______.pt`（最优权重），而不是 `last.pt`。

**练习 3（简答）**：`patience=10` 参数的作用是什么？为什么推荐开启早停？

> 💡 参考答案见下方 code cell。

In [ ]:
# 查看本节练习答案
!cat ./answer/01.04_transfer_learning/answers_training.txt



---

## 第三部分：模型推理与训练效果评估


In [ ]:
from ultralytics import YOLO

# 加载训练后的最优权重
# 路径根据 03.03 中 project/name 的设置而定
model = YOLO("./runs/detect/v10_run1-4/weights/best.pt")

# 查看模型信息
model.info()


## 1. 用训练后的模型检测水果

现在模型已经学会了 5 类水果（cavocado 牛油果、lemon 柠檬、pear 梨、mango 芒果、persimmon 柿子），我们用训练集外的图片测试泛化能力：

In [ ]:
# 用新图片测试（训练集外的图片，检验泛化能力）
result = model.predict(
    source="./images/test_fruit.jpg",
    save_txt=True,       # 保存检测结果为 txt
    save_conf=True,      # 保存置信度
    save=True,           # 保存可视化结果图
    imgsz=416,
    conf=0.25
)

print(f"检测到 {len(result[0].boxes)} 个水果")


In [ ]:
# 可视化检测结果
%matplotlib inline
import matplotlib.pyplot as plt

annotated_frame = result[0].plot()
# BGR → RGB（YOLO 输出是 BGR）
annotated_rgb = annotated_frame[..., ::-1]

plt.figure(figsize=(12, 8))
plt.imshow(annotated_rgb)
plt.axis('off')
plt.title('Fruit Detection after Transfer Learning')
plt.show()


### 训练前后对比

<img src="./images/fruit_detection_result.png" width="500">

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">阶段</th><th align="left">能否识别水果</th><th align="left">说明</th></tr>
<tr><td align="left"><b>训练前</b>（yolov10n.pt 预训练）</td><td align="left">❌ 基本不能</td><td align="left">COCO 80 类不含具体水果种类</td></tr>
<tr><td align="left"><b>训练后</b>（best.pt）</td><td align="left">✅ 准确识别 5 类水果</td><td align="left">迁移训练学会了自定义类别</td></tr>
</table>

这就是迁移学习的威力：**用少量数据 + 短时间训练，让模型学会全新类别的识别**。

## 2. 评估指标解读

训练结果目录下的几张图是评估模型质量的关键：

### 3.1 混淆矩阵（confusion_matrix.png）

<img src="./images/confusion_matrix.png" width="450">

混淆矩阵的读法：

- **对角线**（左上到右下）越亮，说明分类越准；
- **非对角线**有值，说明有混淆（如把"柠檬"误判为"芒果"）；
- 最后一行/列 `background` 表示漏检（有目标没检测到）或误检（把背景当目标）。

### 3.2 PR 曲线（BoxPR_curve.png）

<img src="./images/pr_curve.png" width="450">

PR 曲线的读法：

- **横轴 Recall（召回率）**：所有真实目标中，检测出了多少（越接近 1 越好）；
- **纵轴 Precision（精度）**：检测结果中，有多少是对的（越接近 1 越好）；
- **曲线下面积（mAP）**：综合指标，越接近 1 模型越好。本课程 5 类水果的平均 mAP@0.5 通常能达到 0.8-0.95。

### 3.3 训练曲线（results.png）

<img src="./images/training_results.png" width="500">

观察 loss（box_loss/cls_loss/dfl_loss）是否持续下降、metrics（mAP）是否持续上升。如果 loss 下降但 mAP 不升，可能过拟合了。

---

## 本节练习

**练习 1（选择）**：迁移训练后，应该用哪个权重文件做推理？
- A. `last.pt`（最后一轮）
- B. `best.pt`（验证指标最优）
- C. `yolov10n.pt`（原始预训练）
- D. 任何一个都行

**练习 2（填空）**：混淆矩阵中，对角线越亮说明分类 ______；PR 曲线下面积（mAP）越接近 ______ 说明模型越好。

**练习 3（简答）**：如果训练后发现"柠檬"经常被误判为"芒果"，你会如何改进？（至少提 2 个方案）

> 💡 参考答案见下方 code cell。

In [ ]:
# 查看本节练习答案
!cat ./answer/01.04_transfer_learning/answers_eval.txt
